# Módulo 09 — Juegos Cooperativos y Valor de Shapley

**Objetivos**: Implementar el valor de Shapley iterando sobre permutaciones. Analizar el juego de aeropuerto. Introducir el teorema de Bondareva-Shapley sobre la existencia del core.

In [ ]:
import numpy as np
import itertools
import matplotlib.pyplot as plt
from fractions import Fraction
print('Entorno listo.')

## 1. Función característica y notación

Un juego cooperativo TU (Transferable Utility) es un par (N, v) donde:
- N = {1, ..., n} es el conjunto de jugadores
- v: 2^N → ℝ es la función característica con v(∅) = 0

Representamos v como un diccionario: {frozenset(coalition): value}

In [ ]:
def make_game(v_dict, n):
    """Construye la función característica completa como dict."""
    N = set(range(1, n+1))
    v = {frozenset(): 0}
    for S in range(1, n+1):
        for coalition in itertools.combinations(N, S):
            key = frozenset(coalition)
            sorted_key = tuple(sorted(coalition))
            v[key] = v_dict.get(sorted_key, 0)
    return v

# Juego de mayoría simple con 3 jugadores
majority_dict = {
    (1,): 0, (2,): 0, (3,): 0,
    (1,2): 1, (1,3): 1, (2,3): 1,
    (1,2,3): 1
}
v_maj = make_game(majority_dict, 3)
print('Función característica (mayoría):')
for k, val in sorted(v_maj.items(), key=lambda x: len(x[0])):
    print(f'  v({set(k)}) = {val}')

## 2. Valor de Shapley

**Definición**: φᵢ(v) = (1/n!) · Σ_{ordenaciones σ} [v(Pᵢ(σ) ∪ {i}) − v(Pᵢ(σ))]

donde Pᵢ(σ) es el conjunto de jugadores que preceden a i en la ordenación σ.

**Equivalentemente**: φᵢ(v) = Σ_{S ⊆ N\{i}} [|S|!(n-|S|-1)!/n!] · [v(S∪{i}) − v(S)]

In [ ]:
def shapley_permutations(v, N):
    """
    Calcula el valor de Shapley via permutaciones.
    v: dict {frozenset: value}
    N: lista de jugadores [1, 2, ...]
    Retorna: dict {i: phi_i}
    """
    n = len(N)
    phi = {i: 0.0 for i in N}

    for perm in itertools.permutations(N):
        coalition = set()
        for player in perm:
            before = frozenset(coalition)
            after  = frozenset(coalition | {player})
            marginal = v.get(after, 0) - v.get(before, 0)
            phi[player] += marginal
            coalition.add(player)

    for i in N:
        phi[i] /= len(list(itertools.permutations(N)))  # = n!

    return phi

N = [1, 2, 3]
phi_maj = shapley_permutations(v_maj, N)
print('Valor de Shapley (juego de mayoría):')
for i, val in phi_maj.items():
    print(f'  φ({i}) = {val:.4f}')  # Esperado: 1/3 para todos
print(f'Suma: {sum(phi_maj.values()):.4f} (debe ser v(N) = {v_maj[frozenset(N)]})')

In [ ]:
def shapley_formula(v, N):
    """
    Calcula Shapley con la fórmula de pesos (equivalente pero más eficiente).
    """
    from math import factorial
    n = len(N)
    phi = {i: 0.0 for i in N}

    for i in N:
        others = [j for j in N if j != i]
        for size in range(n):
            for S in itertools.combinations(others, size):
                S_set = frozenset(S)
                S_with_i = frozenset(S) | {i}
                weight = factorial(size) * factorial(n - size - 1) / factorial(n)
                phi[i] += weight * (v.get(S_with_i, 0) - v.get(S_set, 0))

    return phi

phi_maj2 = shapley_formula(v_maj, N)
print('Shapley (fórmula de pesos):')
for i, val in phi_maj2.items():
    print(f'  φ({i}) = {val:.6f}')

## 3. El juego del aeropuerto

Tres aerolíneas comparten el coste de una pista. Los costes de construir la pista que cada una necesita: c₁=1, c₂=3, c₃=5. La pista compartida cuesta max(cᵢ) = 5. El valor de la coalición es lo que ahorra respecto a construir cada una por separado.

In [ ]:
# Costes individuales
c = {1: 1, 2: 3, 3: 5}

def airport_v(coalition):
    """El juego de aeropuerto: v(S) = Σᵢ∈S cᵢ - max(cᵢ for i in S)"""
    if not coalition:
        return 0
    S = list(coalition)
    return sum(c[i] for i in S) - max(c[i] for i in S)

v_air = {frozenset(): 0}
for size in range(1, 4):
    for coal in itertools.combinations([1,2,3], size):
        key = frozenset(coal)
        v_air[key] = airport_v(key)

print('Función v para aeropuerto:')
for k, val in sorted(v_air.items(), key=lambda x: len(x[0])):
    print(f'  v({set(k)}) = {val}')

phi_air = shapley_formula(v_air, [1,2,3])
print('\nValor de Shapley (ahorro por jugador):')
for i, val in phi_air.items():
    cost_share = c[i] - val  # lo que paga cada aerolínea
    print(f'  J{i}: ahorro={val:.3f}, coste asignado={cost_share:.3f}')

## 4. Core e instabilidad

El **core** es el conjunto de repartos (x₁, x₂, x₃) tales que:
- xᵢ ≥ v({i}) para todo i (racionalidad individual)
- xᵢ+xⱼ ≥ v({i,j}) para toda pareja (estabilidad de pares)
- Σxᵢ = v(N) (eficiencia)

El **Teorema de Bondareva-Shapley**: el core no es vacío ↔ el juego es balanceado.

In [ ]:
def in_core(allocation, v, N):
    """Verifica si una asignación está en el core."""
    n = len(N)
    # Eficiencia
    if abs(sum(allocation) - v.get(frozenset(N), 0)) > 1e-6:
        return False, 'No eficiente'
    # Racionalidad individual y de coalición
    for size in range(1, n):
        for S in itertools.combinations(N, size):
            S_set = frozenset(S)
            alloc_S = sum(allocation[N.index(i)] for i in S)
            if alloc_S < v.get(S_set, 0) - 1e-6:
                return False, f'Coalición {set(S)} prefiere separarse: {alloc_S:.3f} < {v.get(S_set,0):.3f}'
    return True, 'Reparto en el core'

# ¿Está el reparto de Shapley en el core del juego de mayoría?
phi_vals = [phi_maj[i] for i in N]
ok, msg = in_core(phi_vals, v_maj, N)
print(f'Juego de mayoría: Shapley en el core? {ok} — {msg}')

# Para el juego de aeropuerto
phi_air_vals = [phi_air[i] for i in N]
ok2, msg2 = in_core(phi_air_vals, v_air, N)
print(f'Juego de aeropuerto: Shapley en el core? {ok2} — {msg2}')

## Ejercicios

**Ejercicio 1**: Calcula el valor de Shapley para el 'mercado de guantes': J1 y J3 tienen guante derecho, J2 tiene guante izquierdo. Un par de guantes vale 1, un solo guante vale 0. v({1,2})=1, v({2,3})=1, todos los demás subconjuntos = 0 excepto v(N)=2.

**Ejercicio 2**: Verifica el axioma de eficiencia y simetría: en el juego de mayoría los tres jugadores son simétricos, por lo tanto φ₁=φ₂=φ₃=v(N)/3. ¿Se cumple?

In [ ]:
# Tu código aquí
